# C2 · Dataset audit

Dùng phần **10.000 identities × 72 captures** mà Microsoft phát hành. Đọc R-UDA và trỏ `DATA_ROOT` đến bản đã giải nén có quyền sử dụng. Release nêu ảnh đã crop. Cần kiểm tra hình học crop trên dữ liệu thực tế. Notebook chỉ đọc các ảnh đã có và ghi audit cùng manifest.

[Nguồn DigiFace-1M](https://github.com/microsoft/DigiFace1M) · [R-UDA](https://github.com/microsoft/DigiFace1M/blob/main/LICENSE)

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = DRIVE / "C2_Base"
WORK = DRIVE / "C2_work"
DATA_ROOT = Path("/content/digiface72")  # Extracted, authorized 72-capture partition
MANIFEST = WORK / "manifests/digiface72.csv"

assert (REPO / "configs/pilot.yaml").is_file(), f"Set REPO to the copied C2_Base source folder: {REPO}"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

In [ ]:
AUDIT = WORK / "manifests/digiface72_audit.json"
ARCHIVES = None  # Optional Path to JSON mapping part directory to archive and sha256.
REBUILD_MANIFEST = False
assert DATA_ROOT.is_dir(), f"Set DATA_ROOT to the extracted 72-capture partition: {DATA_ROOT}"

if REBUILD_MANIFEST or not (MANIFEST.is_file() and AUDIT.is_file()):
    args = ["--data-root", DATA_ROOT, "--manifest", MANIFEST, "--audit", AUDIT]
    if ARCHIVES is not None:
        args += ["--archives", ARCHIVES]
    run("scripts.audit_dataset", *args)
report = json.loads(AUDIT.read_text(encoding="utf-8"))
assert report["status"] == "PASS", report
print(json.dumps(report, indent=2))

Kiểm tra `pilot_images=40000` và `split_ids` lần lượt 6000/1000/1000/2000. Manifest lưu capture hashes và identity assignment trước khi tạo pairs. Đặt `DATA_ROOT` giống nhau trong các notebook training/evaluation.